In [ ]:
import os
import pandas as pd
import datetime
import decimal
import uuid
from pyspark.sql import functions as F
from pyspark.sql.functions import col, date_trunc, when
from src.utils.config import setup_clickhouse_client
from src.utils.kafka import create_kafka_consumer
from src.utils.spark import get_spark

spark = get_spark()

ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')
client = setup_clickhouse_client(ch_user, ch_password, ch_host)
boostrap_servers = os.getenv('KAFKA_BOOTSTRAP_SERVERS')

# consumer_instance = create_kafka_consumer(
#     client_id='flight_processor',
#     bootstrap_servers=boostrap_servers,
#     topic_name='raw_aviation_data',
# )

In [ ]:
def convert_row(row):
    return [
        x.isoformat() if isinstance(x, datetime.datetime)
        else float(x) if isinstance(x, decimal.Decimal)
        else str(x) if isinstance(x, uuid.UUID) 
        else ','.join(map(str, x)) if isinstance(x, list)
        else x
        for x in row
    ]

In [ ]:
# Fetch flights data
flights_query = "SELECT * FROM aviation_flights order by (dep_scheduled_time, dep_actual_time, arr_scheduled_time, arr_actual_time)"
flights_result = client.query(flights_query).result_rows
flights_columns = client.query(flights_query).column_names
flights_result = [convert_row(row) for row in flights_result]

# Create spark DataFrame
flights_pdf = pd.DataFrame(flights_result, columns=flights_columns)

print(f"Total flight records retrieved: {len(flights_pdf)}")
flights_pdf.head(5)

In [ ]:
weather_query = "SELECT " \
"date_observed, current_temp, pressure_hPa, humidity_pct, max_temp, min_temp, wind_speed_ms, " \
"wind_deg, cloudiness, rain_1h, rain_3h, weather_main, weather_desc " \
"FROM historical_weather_data " \
"order by date_observed"

weather_result = client.query(weather_query).result_rows

weather_columns = client.query(weather_query).column_names
weather_result = [convert_row(row) for row in weather_result]

print(f"Weather columns: {weather_columns}, Weather result sample: {weather_result[:2]}")

# Use pandas as intermediary for weather (Windows Spark serialization workaround)
weather_pdf = pd.DataFrame(weather_result, columns=weather_columns)

print(f"Total weather records retrieved: {len(weather_pdf)}")
weather_pdf.head(5)

In [ ]:
print('Prepared flights and weather data for merging.')

# Filter out cancelled and unknown status flights
valid_statuses = ['cancelled', 'unknown']
flights_filtered = flights_pdf.query(f"status not in {valid_statuses}") \
                    .drop_duplicates(subset=['flight_type', 'status', 'iata_number', 'airline_name', 
                                            'dep_scheduled_time', 'arr_scheduled_time', 'dep_actual_time', 'arr_actual_time']) \
                    .copy()

print(f"Flight records after filtering: {len(flights_filtered)}")

In [ ]:
# Prepare weather data: round date_observed down to the nearest hour
weather_prep = weather_pdf.copy()
weather_prep['weather_hour'] = pd.to_datetime(weather_prep['date_observed']).dt.floor('h')

# Deduplicate weather data: keep only one record per hour (average numeric columns, keep first for others)
weather_prep = weather_prep.groupby('weather_hour', as_index=False) \
    .agg({ 
        col: 'mean' if weather_prep[col].dtype in ['float64', 'int64'] else 'first'
        for col in weather_prep.columns if col != 'weather_hour'
    })

print(f"Weather records after removing duplicates by hour: {len(weather_prep)}")

In [ ]:
# Create hour columns for both departure and arrival times
flights_filtered['dep_hour'] = pd.to_datetime(flights_filtered['dep_scheduled_time']).dt.floor('h')
flights_filtered['arr_hour'] = pd.to_datetime(flights_filtered['arr_scheduled_time']).dt.floor('h')

# Split into departure and arrival flights for conditional merging
departure_flights = flights_filtered.query("flight_type == 'departure'").copy()
arrival_flights = flights_filtered.query("flight_type == 'arrival'").copy()

print(f"\nDeparture flights: {len(departure_flights)}, Arrival flights: {len(arrival_flights)}")

In [ ]:
# Merge departure flights with weather
# Departure: join dep_hour with weather_hour (not exceeding scheduled time means weather_hour <= dep_hour)
if len(departure_flights) > 0:
    merged_departures = pd.merge(
        departure_flights,
        weather_prep,
        left_on='dep_hour',
        right_on='weather_hour',
        how='left'
    )
    print(f"Merged departure flights: {len(merged_departures)}")

# Merge arrival flights with weather
# Arrival: join arr_hour with weather_hour (same hour)
if len(arrival_flights) > 0:
    merged_arrivals = pd.merge(
        arrival_flights,
        weather_prep,
        left_on='arr_hour',
        right_on='weather_hour',
        how='left'
    )
    print(f"Merged arrival flights: {len(merged_arrivals)}")

# Combine both datasets
if len(merged_departures) > 0 and len(merged_arrivals) > 0:
    combined_flights = pd.concat([merged_departures, merged_arrivals], ignore_index=True, verify_integrity=True) \
            .sort_values(by=['dep_scheduled_time', 'dep_actual_time', 'arr_scheduled_time', 'arr_actual_time'], na_position='last')
    # print(combined_flights.columns)
    combined_flights.drop(['id_x', 'insert_time_x', 'iata_number', 'airline_name', 'codeshared_airline', 'codeshared_flight_number', 'id_y', 'insert_time_y'], axis=1, inplace=True)
else:
    combined_flights = pd.DataFrame()

print(f"\n Combined flights with weather data: {len(combined_flights)} rows")
combined_flights.head(3)

Merged departure flights: 63390
Merged arrival flights: 62674
Index(['id_x', 'insert_time_x', 'flight_type', 'status', 'iata_number',
       'airline_name', 'dep_scheduled_time', 'dep_actual_time',
       'dep_delay_mins', 'arr_scheduled_time', 'arr_actual_time',
       'arr_delay_mins', 'codeshared_airline', 'codeshared_flight_number',
       'dep_hour', 'arr_hour', 'weather_hour', 'id_y', 'insert_time_y',
       'date_observed', 'current_temp', 'feels_like_temp', 'pressure_hPa',
       'humidity_pct', 'max_temp', 'min_temp', 'wind_speed_ms', 'wind_deg',
       'cloudiness_pct', 'rain_1h', 'rain_3h', 'weather_main', 'weather_desc'],
      dtype='str')

 Combined flights with weather data: 126064 rows


,flight_type,status,dep_scheduled_time,dep_actual_time,dep_delay_mins,arr_scheduled_time,arr_actual_time,arr_delay_mins,dep_hour,arr_hour,...,humidity_pct,max_temp,min_temp,wind_speed_ms,wind_deg,cloudiness_pct,rain_1h,rain_3h,weather_main,weather_desc
63390,arrival,landed,2026-02-27T14:35:00,2026-02-27T14:57:00,22,2026-03-01T00:01:00,2026-02-28T23:35:00,0,2026-02-27 14:00:00,2026-03-01 00:00:00,...,90.0,25.92,27.18,2.57,40.0,75.0,0.0,0.0,Clouds,broken clouds
63391,arrival,landed,2026-02-28T00:45:00,2026-02-28T01:18:00,33,2026-03-01T09:45:00,2026-03-01T09:33:00,0,2026-02-28 00:00:00,2026-03-01 09:00:00,...,64.0,30.92,31.97,4.12,110.0,75.0,0.0,0.0,Clouds,broken clouds
63392,arrival,landed,2026-02-28T01:35:00,2026-02-28T02:03:00,29,2026-03-01T09:45:00,2026-03-01T09:25:00,0,2026-02-28 01:00:00,2026-03-01 09:00:00,...,64.0,30.92,31.97,4.12,110.0,75.0,0.0,0.0,Clouds,broken clouds


In [ ]:
# Data validation and quality checks for weather data (Ignore for now)

print("\n" + "=" * 50)
print("WEATHER DATA QUALITY CHECKS")
print("=" * 50)

# Check for nulls in key columns
print("\nNull counts in key columns:")
null_check_weather = weather_pdf.select([
    F.count(when(col(c).isNull(), 1)).alias(f"{c}_nulls") 
    for c in ['date_observed', 'current_temp', 'humidity_pct', 'wind_speed_ms']
])
null_check_weather.show()

# Check data types and date ranges
print("\nWeather date range:")
date_range_weather = weather_pdf.select(
    F.min('date_observed').alias('earliest_obs'),
    F.max('date_observed').alias('latest_obs'),
    F.count('*').alias('total_records')
)
date_range_weather.show()

# Check temperature ranges (sanity check)
print("\nTemperature statistics (Celsius):")
weather_pdf.select(
    F.min('current_temp').alias('min_temp'),
    F.max('current_temp').alias('max_temp'),
    F.avg('current_temp').alias('avg_temp'),
    F.stddev('current_temp').alias('std_temp')
).show()

# Check humidity and wind ranges
print("\nHumidity and wind statistics:")
weather_pdf.select(
    F.min('humidity_pct').alias('min_humidity'),
    F.max('humidity_pct').alias('max_humidity'),
    F.min('wind_speed_ms').alias('min_wind'),
    F.max('wind_speed_ms').alias('max_wind')
).show()

print(f"\nTotal weather records: {weather_pdf.count()}")


In [ ]:
# Data validation and quality checks for flights data

print("=" * 50)
print("FLIGHTS DATA QUALITY CHECKS")
print("=" * 50)

# Check for nulls in key columns
print("\nNull counts in key columns:")
null_check_flights = flights_pdf.select([
    F.count(when(col(c).isNull(), 1)).alias(f"{c}_nulls") 
    for c in ['iata_number', 'dep_scheduled_time', 'arr_scheduled_time', 'dep_delay_mins']
])
null_check_flights.show()

# Check data types and date ranges
print("\nFlights date range:")
date_range_flights = flights_pdf.select(
    F.min('dep_scheduled_time').alias('earliest_dep'),
    F.max('dep_scheduled_time').alias('latest_dep'),
    F.min('arr_scheduled_time').alias('earliest_arr'),
    F.max('arr_scheduled_time').alias('latest_arr')
)
date_range_flights.show()

# Check flight status distribution
print("\nFlight status distribution:")
flights_pdf.select('status').groupBy('status').count().show()

# Check flight type distribution (arrival vs departure)
print("\nFlight type distribution:")
flights_pdf.select('flight_type').groupBy('flight_type').count().show()

print(f"\nTotal flight records: {flights_pdf.count()}")
